# Lab 3-04: Run & Benchmark a Local NVIDIA NIM Container (Optional GPU Lab)

**Chapter 3: Software Development for Generative AI and LLMs**  
**Lab Type:** Local GPU Lab (with Built-In Local Mock NIM Server for CPU/Cloud Notebooks) | **Estimated Time:** 45 minutes  
**Exam Domains Covered:** Software Development (24%), Core ML & AI Knowledge (30%)

---

## Learning Objectives
By the end of this lab, you will be able to:
1. Explain every flag in the canonical **`docker run`** command used to deploy a self-hosted **NVIDIA NIM** container (`--gpus all`, `-e NGC_API_KEY`, `--shm-size=16GB`, `-v "$LOCAL_NIM_CACHE:/opt/nim/.cache"`, `-p 8000:8000`).
2. Query a local NIM microservice's operational endpoints (`/v1/health/ready`, `/v1/health/live`, `/v1/models`, and `/v1/chat/completions`).
3. Switch an existing OpenAI-compatible client from NVIDIA's cloud catalog (`https://integrate.api.nvidia.com/v1`) to a self-hosted NIM container (`http://localhost:8000/v1`) with **zero application logic changes**.
4. Repeat the Lab 3-03 streaming & latency benchmark against `http://localhost:8000/v1`.

---
## 1. Conceptual Foundation: Containerization, Docker & NVIDIA NIM

In **Chapter 3 (*Containerization and Deployment with Docker and NVIDIA NIM*)**, the book explains why enterprise teams package AI workloads in **Docker containers** with the **NVIDIA Container Toolkit**:
- **Reproducibility ("Dependency Hell" Prevention):** Bundles exact CUDA libraries, TensorRT-LLM engines, Python packages, and system dependencies into an immutable image.
- **GPU Passthrough (`--gpus all`):** Standard Docker containers are isolated from host GPUs. Installing the **NVIDIA Container Toolkit (`nvidia-container-toolkit`)** allows Docker to expose host NVIDIA GPUs inside the container via `--gpus all` (or `--gpus '"device=0,1"'`).

### Anatomy of the Canonical NVIDIA NIM `docker run` Command (Exam Favorite!)

```bash
export NGC_API_KEY="nvapi-..."
export LOCAL_NIM_CACHE=~/.cache/nim
mkdir -p "$LOCAL_NIM_CACHE"

docker run -it --rm \
  --name=llama3-8b-nim \
  --gpus all \
  --shm-size=16GB \
  -e NGC_API_KEY=$NGC_API_KEY \
  -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
  -u $(id -u) \
  -p 8000:8000 \
  nvcr.io/nim/meta/llama3-8b-instruct:latest
```

| Flag | Purpose & Why It Matters on the Exam |
| :--- | :--- |
| `--gpus all` | Exposes host NVIDIA GPU(s) to the container via the NVIDIA Container Toolkit |
| `--shm-size=16GB` | Allocates shared memory (`/dev/shm`) required by **NCCL / TensorRT-LLM** for inter-process communication |
| `-e NGC_API_KEY=$NGC_API_KEY` | Authenticates the container with NGC to pull the optimal **TensorRT-LLM engine profile** for the detected GPU architecture (e.g., H100 vs. A100 vs. L40S) |
| `-v "$LOCAL_NIM_CACHE:/opt/nim/.cache"` | Mounts a persistent host volume so downloaded model weights and compiled TensorRT-LLM engines are **cached across container restarts** (avoiding a 15 GB re-download!) |
| `-p 8000:8000` | Maps host port `8000` to the container's internal HTTP server port `8000` |

In [1]:
import os
import time
import json
import socket
import threading
import urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

LOCAL_NIM_PORT = 8000
LOCAL_NIM_BASE_URL = f"http://127.0.0.1:{LOCAL_NIM_PORT}/v1"


def is_port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


print(f"Checking if a local NIM Docker container is already listening on port {LOCAL_NIM_PORT}: {is_port_in_use(LOCAL_NIM_PORT)}")

Checking if a local NIM Docker container is already listening on port 8000: False


---
## 2. Launching a Local NIM Endpoint (or Automatic Local NIM Emulator on Port 8000)

If you are running this notebook on a machine with an NVIDIA GPU (24GB+ VRAM) and have launched the `nvcr.io/nim/meta/llama3-8b-instruct:latest` container on port `8000`, the cells below will communicate directly with your live Docker container!

If no container is currently listening on port `8000` (for example, on a laptop or standard cloud notebook), the cell below automatically starts a lightweight, spec-compliant **Local NIM HTTP Microservice Emulator** on `127.0.0.1:8000` (or the next free port) implementing:
- `GET /v1/health/live` & `GET /v1/health/ready`
- `GET /v1/models`
- `POST /v1/chat/completions` (supporting both `stream=False` JSON and `stream=True` Server-Sent Events `data: {...}\n\n`)

This lets you execute real HTTP health checks and OpenAI SDK streaming calls against `localhost` right inside this notebook!

In [2]:
class MockNIMHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # Keep notebook output clean

    def do_GET(self):
        if self.path in ("/v1/health/ready", "/v1/health/live"):
            body = json.dumps({"status": "ready", "service": "nim-llm", "engine": "tensorrt_llm"}).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
        elif self.path == "/v1/models":
            body = json.dumps({
                "object": "list",
                "data": [{
                    "id": "meta/llama3-8b-instruct",
                    "object": "model",
                    "owned_by": "nvidia-nim-local",
                    "max_model_len": 8192,
                }]
            }).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
        else:
            self.send_response(404)
            self.end_headers()

    def do_POST(self):
        if self.path == "/v1/chat/completions":
            length = int(self.headers.get("Content-Length", 0))
            payload = json.loads(self.rfile.read(length).decode("utf-8"))
            is_stream = payload.get("stream", False)
            model_id = payload.get("model", "meta/llama3-8b-instruct")
            tokens = (
                "Local NVIDIA NIM container responded from localhost! "
                "TensorRT-LLM in-flight batching and paged KV caching are active."
            ).split()

            if not is_stream:
                time.sleep(0.08)
                resp_obj = {
                    "id": "chatcmpl-nim-local-001",
                    "object": "chat.completion",
                    "model": model_id,
                    "choices": [{
                        "index": 0,
                        "message": {"role": "assistant", "content": " ".join(tokens)},
                        "finish_reason": "stop",
                    }],
                    "usage": {"prompt_tokens": 24, "completion_tokens": len(tokens), "total_tokens": 24 + len(tokens)},
                }
                body = json.dumps(resp_obj).encode("utf-8")
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(body)
            else:
                self.send_response(200)
                self.send_header("Content-Type", "text/event-stream")
                self.send_header("Cache-Control", "no-cache")
                self.end_headers()
                time.sleep(0.035)  # Local LAN/loopback TTFT (~35 ms)
                for i, tok in enumerate(tokens):
                    chunk_obj = {
                        "id": "chatcmpl-nim-local-stream",
                        "object": "chat.completion.chunk",
                        "model": model_id,
                        "choices": [{
                            "index": 0,
                            "delta": {"content": tok + (" " if i < len(tokens) - 1 else "")},
                            "finish_reason": None,
                        }],
                    }
                    self.wfile.write(f"data: {json.dumps(chunk_obj)}\n\n".encode("utf-8"))
                    self.wfile.flush()
                    time.sleep(0.008)  # ~8 ms local ITL
                done_chunk = {
                    "id": "chatcmpl-nim-local-stream",
                    "object": "chat.completion.chunk",
                    "model": model_id,
                    "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
                }
                self.wfile.write(f"data: {json.dumps(done_chunk)}\n\ndata: [DONE]\n\n".encode("utf-8"))
                self.wfile.flush()


if not is_port_in_use(LOCAL_NIM_PORT):
    server = HTTPServer(("127.0.0.1", LOCAL_NIM_PORT), MockNIMHandler)
    server_thread = threading.Thread(target=server.serve_forever, daemon=True)
    server_thread.start()
    print(f"Started local OpenAI-compatible NIM endpoint emulator at http://127.0.0.1:{LOCAL_NIM_PORT}/v1")
else:
    print(f"Connected to existing server on http://127.0.0.1:{LOCAL_NIM_PORT}/v1")

Started local OpenAI-compatible NIM endpoint emulator at http://127.0.0.1:8000/v1


---
## 3. Probing NIM Health & Model Discovery Endpoints

In production Kubernetes or Docker deployments, orchestrators poll `/v1/health/ready` before routing traffic to a newly started NIM container (since compiling or loading a TensorRT-LLM engine into GPU VRAM can take 30–90 seconds at startup). Let's query both `/v1/health/ready` and `/v1/models`.

In [3]:
def inspect_local_nim_endpoints(base_url: str = f"http://127.0.0.1:{LOCAL_NIM_PORT}/v1"):
    for path in ["/health/ready", "/models"]:
        url = base_url + path
        with urllib.request.urlopen(url, timeout=5) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            print(f"GET {url} (HTTP {resp.status}):")
            print(json.dumps(data, indent=2))
            print("-" * 60)

inspect_local_nim_endpoints()

GET http://127.0.0.1:8000/v1/health/ready (HTTP 200):
{
  "status": "ready",
  "service": "nim-llm",
  "engine": "tensorrt_llm"
}
------------------------------------------------------------
GET http://127.0.0.1:8000/v1/models (HTTP 200):
{
  "object": "list",
  "data": [
    {
      "id": "meta/llama3-8b-instruct",
      "object": "model",
      "owned_by": "nvidia-nim-local",
      "max_model_len": 8192
    }
  ]
}
------------------------------------------------------------


---
## 4. Benchmarking Streaming Latency Against `http://localhost:8000/v1`

Now let's run the exact same streaming latency benchmark from **Lab 3-03**, pointing at `http://127.0.0.1:8000/v1`. Notice how self-hosting a NIM container inside your local VPC/workstation eliminates WAN network round-trip overhead!

In [4]:
def benchmark_local_nim_stream(base_url: str = f"http://127.0.0.1:{LOCAL_NIM_PORT}/v1"):
    req_payload = json.dumps({
        "model": "meta/llama3-8b-instruct",
        "messages": [{"role": "user", "content": "Verify local NIM container status."}],
        "stream": True,
        "max_tokens": 64,
    }).encode("utf-8")

    req = urllib.request.Request(
        f"{base_url}/chat/completions",
        data=req_payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    timestamps = []
    pieces = []
    t_send = time.perf_counter()

    with urllib.request.urlopen(req, timeout=10) as resp:
        for raw_line in resp:
            line = raw_line.decode("utf-8").strip()
            if not line.startswith("data: "):
                continue
            payload_str = line[len("data: "):]
            if payload_str == "[DONE]":
                break
            chunk = json.loads(payload_str)
            delta = chunk["choices"][0].get("delta", {})
            if "content" in delta and delta["content"]:
                timestamps.append(time.perf_counter())
                pieces.append(delta["content"])

    ttft_ms = (timestamps[0] - t_send) * 1000.0
    e2e_ms = (timestamps[-1] - t_send) * 1000.0
    itls_ms = [(timestamps[i] - timestamps[i - 1]) * 1000.0 for i in range(1, len(timestamps))]
    mean_itl_ms = sum(itls_ms) / max(1, len(itls_ms))

    print("=== LOCAL NIM CONTAINER STREAMING BENCHMARK (http://127.0.0.1:8000/v1) ===")
    print(f"Streamed Response : {''.join(pieces)}")
    print(f"Tokens Received   : {len(timestamps)}")
    print(f"Local TTFT        : {ttft_ms:.1f} ms")
    print(f"Local Mean ITL    : {mean_itl_ms:.2f} ms")
    print(f"Local E2E Latency : {e2e_ms:.1f} ms")

benchmark_local_nim_stream()

=== LOCAL NIM CONTAINER STREAMING BENCHMARK (http://127.0.0.1:8000/v1) ===
Streamed Response : Local NVIDIA NIM container responded from localhost! TensorRT-LLM in-flight batching and paged KV caching are active.
Tokens Received   : 16
Local TTFT        : 36.4 ms
Local Mean ITL    : 8.31 ms
Local E2E Latency : 161.0 ms


---
## 5. Hands-On Exercise: Build a NIM `docker run` Command Generator

Write a Python function `generate_nim_docker_cmd(image_repo, host_port, gpu_spec, cache_dir)` that validates inputs and generates the exact production `docker run` command string for deploying an NVIDIA NIM container.

In [5]:
# Solution / Exercise: NIM Docker Command Generator
def generate_nim_docker_cmd(
    image_repo: str = "nvcr.io/nim/meta/llama3-8b-instruct:latest",
    host_port: int = 8000,
    gpu_spec: str = "all",
    cache_dir: str = "~/.cache/nim",
) -> str:
    return (
        f"docker run -it --rm \\\n"
        f"  --gpus {gpu_spec} \\\n"
        f"  --shm-size=16GB \\\n"
        f"  -e NGC_API_KEY=$NGC_API_KEY \\\n"
        f"  -v \"{cache_dir}:/opt/nim/.cache\" \\\n"
        f"  -p {host_port}:8000 \\\n"
        f"  {image_repo}"
    )

print(generate_nim_docker_cmd())

docker run -it --rm \
  --gpus all \
  --shm-size=16GB \
  -e NGC_API_KEY=$NGC_API_KEY \
  -v "~/.cache/nim:/opt/nim/.cache" \
  -p 8000:8000 \
  nvcr.io/nim/meta/llama3-8b-instruct:latest


---
## 6. NCA-GENL Chapter 3 Review Takeaways

- **Why NVIDIA NIM?** Pre-built, self-contained inference containers that automatically select the optimal **TensorRT-LLM** engine and expose standard OpenAI-compatible REST endpoints (`/v1/chat/completions`, `/v1/embeddings`).
- **Key `docker run` Flags:** `--gpus all` (GPU access via NVIDIA Container Toolkit), `-e NGC_API_KEY` (engine download authentication), `-v "$LOCAL_NIM_CACHE:/opt/nim/.cache"` (persistent weight/engine cache), and `-p 8000:8000` (port mapping).
- **Zero-Code Migration:** Switching from `https://integrate.api.nvidia.com/v1` to a local NIM container only requires changing `base_url="http://localhost:8000/v1"`.